# MODIS \u2194 VIIRS Fire Detection Harmonization

Implements the four-layer methodology in `docs/HARMONIZATION_METHODOLOGY.md` on the Cambodia-filtered FIRMS parquet files:

1. **Layer 0** -- Schema unification (single unified view).
2. **Layer 1** -- Semantic normalization (confidence rank/norm, fire-type filter, day/night stratification).
3. **Layer 2** -- Spatial-temporal alignment (grid aggregation + per-grid deduplication, pixel-area normalization).
4. **Layer 3** -- Radiometric cross-calibration (FRP regression, brightness-temperature regressions, sensor match rates).

Runs the six-item validation checklist and stores `fires_unified.parquet`.


In [ ]:
import pandas as pd
import duckdb
import numpy as np
import matplotlib.pyplot as plt

MODIS_PATH = "firms_parquet/cambodia_modis.parquet"
VIIRS_PATH = "firms_parquet/cambodia_viirs.parquet"

con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial")

modis = pd.read_parquet(MODIS_PATH)
viirs = pd.read_parquet(VIIRS_PATH)

print(f"MODIS raw : {len(modis):,} rows")
print(f"VIIRS raw : {len(viirs):,} rows")


In [ ]:
def parse_acq_time(s):
    return pd.to_datetime(s.str.zfill(4), format='%H%M').dt.time

modis_proc = pd.DataFrame({
    'sensor': 'MODIS',
    'satellite': modis['satellite'],
    'instrument': modis['instrument'],
    'acq_date': pd.to_datetime(modis['acq_date']).dt.date,
    'acq_time_utc': parse_acq_time(modis['acq_time']),
    'latitude': modis['latitude'],
    'longitude': modis['longitude'],
    'bright_t4': modis['brightness'],
    'bright_t11': modis['bright_t31'],
    'frp': modis['frp'],
    'daynight': modis['daynight'],
    'type': modis['type'],
    'confidence_native': modis['confidence'].astype(float),
    'track_km': modis['track'] * 111.0,
    'scan_km': modis['scan'] * 111.0,
})

viirs_proc = pd.DataFrame({
    'sensor': 'VIIRS',
    'satellite': viirs['satellite'],
    'instrument': viirs['instrument'],
    'acq_date': pd.to_datetime(viirs['acq_date']).dt.date,
    'acq_time_utc': parse_acq_time(viirs['acq_time']),
    'latitude': viirs['latitude'],
    'longitude': viirs['longitude'],
    'bright_t4': viirs['bright_ti4'],
    'bright_t11': viirs['bright_ti5'],
    'frp': viirs['frp'],
    'daynight': viirs['daynight'],
    'type': viirs['type'],
    'confidence_native': viirs['confidence'].map({'l': 20, 'n': 60, 'h': 90}),
    'track_km': viirs['track'] * 111.0,
    'scan_km': viirs['scan'] * 111.0,
})

fires_raw = pd.concat([modis_proc, viirs_proc], ignore_index=True)
fires_raw.head()


In [ ]:
def add_confidence_columns(df, sensor):
    if sensor == 'MODIS':
        rank = df['confidence_native'].map(lambda c: 1 if c < 30 else (2 if c <= 79 else 3)).astype(int)
        norm = df['confidence_native'] / 100.0
    else:
        rank = df['confidence_native'].map({20: 1, 60: 2, 90: 3}).astype(int)
        norm = df['confidence_native'].map({20: 0.20, 60: 0.60, 90: 0.90})
    df['confidence_rank'] = rank
    df['confidence_norm'] = norm
    return df

modis_p = add_confidence_columns(modis_proc.copy(), 'MODIS')
viirs_p = add_confidence_columns(viirs_proc.copy(), 'VIIRS')
fires_raw = pd.concat([modis_p, viirs_p], ignore_index=True)

before = fires_raw.groupby('sensor').size()
fires = fires_raw[(fires_raw['confidence_rank'] >= 2) & (fires_raw['type'] == 0)].copy()
after = fires.groupby('sensor').size()
print(f"Before filtering: MODIS={before.get('MODIS',0):,}  VIIRS={before.get('VIIRS',0):,}")
print(f"After  filtering: MODIS={after.get('MODIS',0):,}  VIIRS={after.get('VIIRS',0):,}")
print(f"Drop %: MODIS={100*(1-after.get('MODIS',0)/before.get('MODIS',1)):.1f}%  VIIRS={100*(1-after.get('VIIRS',0)/before.get('VIIRS',1)):.1f}%")

# Day/night stratification option
fires_day = fires[fires['daynight'] == 'D'].copy()
fires_night = fires[fires['daynight'] == 'N'].copy()
print(f"Day detections : {len(fires_day):,}")
print(f"Night detections: {len(fires_night):,}")


In [ ]:
fires['pixel_area_km2'] = fires['scan_km'] * fires['track_km']
fires['frp_density_mw_per_km2'] = fires['frp'] / fires['pixel_area_km2']
print(fires[['pixel_area_km2','frp_density_mw_per_km2']].describe())


In [ ]:
fires['grid_lat'] = (fires['latitude'] // 0.01) * 0.01
fires['grid_lon'] = (fires['longitude'] // 0.01) * 0.01

fires_gridded = (
    fires.groupby(['grid_lat', 'grid_lon', 'acq_date', 'sensor'])
    .agg(detection_count=('frp', 'size'), frp_sum=('frp', 'sum'), confidence_max=('confidence_rank', 'max'), bright_t4=('bright_t4', 'mean'))
    .reset_index()
)
print(f"Gridded rows : {len(fires_gridded):,}")
fires_gridded.head()


In [ ]:
filtered = fires_gridded.sort_values(['grid_lat','grid_lon','acq_date','confidence_max','frp_sum'], ascending=[True,True,True,False,False])
fires_dedup = filtered.drop_duplicates(subset=['grid_lat','grid_lon','acq_date'], keep='first')

print(f"Gridded cell-days : {len(fires_gridded):,}")
print(f"Deduped cell-days   : {len(fires_dedup):,}")
print(f"Reduction           : {100*(1-len(fires_dedup)/len(fires_gridded)):.1f}%")

modis_cells = fires_gridded[fires_gridded['sensor']=='MODIS'][['grid_lat','grid_lon','acq_date']].copy()
modis_cells['modis'] = True
viirs_cells = fires_gridded[fires_gridded['sensor']=='VIIRS'][['grid_lat','grid_lon','acq_date']].copy()
viirs_cells['viirs'] = True
cross = modis_cells.merge(viirs_cells, on=['grid_lat','grid_lon','acq_date'], how='outer')
print(cross[['modis','viirs']].notna().sum().to_dict())


In [ ]:
fires['detection_timestamp'] = pd.to_datetime(fires['acq_date'].astype(str) + ' ' + fires['acq_time_utc'].astype(str))
print(fires[['acq_date','acq_time_utc','detection_timestamp']].head(3))


In [ ]:
# --- FRP regression: matched grid-day cells where both sensors detected fire ---
m1 = fires_gridded[fires_gridded['sensor']=='MODIS'].rename(columns={'grid_lat':'glat','grid_lon':'glon','acq_date':'adate','frp_sum':'modis_frp_sum','detection_count':'modis_count'})
m2 = fires_gridded[fires_gridded['sensor']=='VIIRS'].rename(columns={'grid_lat':'glat','grid_lon':'glon','acq_date':'adate','frp_sum':'viirs_frp_sum','detection_count':'viirs_count'})
matched = m1.merge(m2, on=['glat','glon','adate'])
print(f"Matched grid-day cells (MODIS + VIIRS): {len(matched):,}")

x = matched['modis_frp_sum'].to_numpy()
y = matched['viirs_frp_sum'].to_numpy()
a, b = np.polyfit(x[y > 0], y[y > 0], 1)
print(f"FRP calibration: FRP_viirs_sum = {a:.4f} * FRP_modis_sum + {b:.2f}")
print(f"Median VIIRS/MODIS FRP-sum ratio: {np.median(y[x > 0] / np.maximum(x[x > 0], 1e-9)):.2f}")

# --- Brightness-temperature regression on matched detections ---
bt = matched.merge(m1[['glat','glon','adate','bright_t4']].rename(columns={'bright_t4':'modis_t4'}), on=['glat','glon','adate'], how='left')
bt = bt.merge(m2[['glat','glon','adate','bright_t4']].rename(columns={'bright_t4':'viirs_ti4'}), on=['glat','glon','adate'], how='left')
bt_valid = bt.dropna(subset=['modis_t4', 'viirs_ti4'])
print(f"Matched detections for brightness regression: {len(bt_valid):,}")
if len(bt_valid) > 0:
    print(f"Pearson r(bright_t4_modis, bright_ti4_viirs) = {bt_valid['modis_t4'].corr(bt_valid['viirs_ti4']):.3f}")
    print(np.polyfit(bt_valid['viirs_ti4'], bt_valid['modis_t4'], 1))

# --- Sensor match rates (same day, within ~1 km) ---
con.register('fires_unified_df', fires)
modis_rate = con.execute(r"""
SELECT f.sensor, COUNT(*) AS n FROM fires_unified_df f LEFT JOIN fires_unified_df v ON f.sensor = 'MODIS' AND v.sensor = 'VIIRS' AND v.acq_date = f.acq_date AND ST_DWithin(ST_Point(f.longitude, f.latitude), ST_Point(v.longitude, v.latitude), 0.01) GROUP BY f.sensor
""").fetchdf()
print(modis_rate)

viirs_rate = con.execute(r"""
SELECT v.sensor, COUNT(*) AS n FROM fires_unified_df v LEFT JOIN fires_unified_df f ON v.sensor = 'VIIRS' AND f.sensor = 'MODIS' AND f.acq_date = v.acq_date AND ST_DWithin(ST_Point(v.longitude, v.latitude), ST_Point(f.longitude, f.latitude), 0.01) GROUP BY v.sensor
""").fetchdf()
print(viirs_rate)


In [ ]:
pre_2012 = fires[fires['acq_date'] < pd.to_datetime('2012-01-01').date()]
post_2012 = fires[fires['acq_date'] >= pd.to_datetime('2012-01-01').date()]
print("Detections by sensor period (post type/conf filter):")
print(pre_2012.groupby('sensor').size())
print(post_2012.groupby('sensor').size())


In [ ]:
overlap = fires[fires['acq_date'] >= pd.to_datetime('2012-01-01').date()]
annual = overlap.groupby(['sensor', pd.Grouper(key='detection_timestamp', freq='YE')]).size().reset_index(name='n')
print(annual.head(10))
m_annual = annual[annual['sensor']=='MODIS']['n'].mean()
v_annual = annual[annual['sensor']=='VIIRS']['n'].mean()
print(f"Mean annual detections on overlap: MODIS={m_annual:,.0f}  VIIRS={v_annual:,.0f}  ratio={v_annual/m_annual:.1f}x (guideline ~6x)")


In [ ]:
monthly = fires.groupby(fires['acq_date'].map(lambda d: d.month)).size()
print("Detections by month:")
print(monthly)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(1, 13), [monthly.get(m, 0) for m in range(1, 13)])
ax.set_xlabel("Month")
ax.set_ylabel("Detections")
ax.set_title("Cambodia fire detections by month (harmonized)")
ax.set_xticks(range(1, 13))
plt.show()


In [ ]:
combined = fires_gridded[['grid_lat','grid_lon','acq_date','frp_sum']].sort_values('frp_sum', ascending=False).head(5)
print(combined.to_string(index=False))


In [ ]:
drop_modis = 100*(1-after.get('MODIS',0)/before.get('MODIS',1))
drop_viirs = 100*(1-after.get('VIIRS',0)/before.get('VIIRS',1))
print(f"Observed drop: MODIS={drop_modis:.1f}%  VIIRS={drop_viirs:.1f}%")
print("Methodology guideline: ~25-35% for both. A large asymmetry would indicate broken confidence mapping.")


In [ ]:
ratio_raw = len(modis_proc) / len(viirs_proc)
ratio_cell = len(m1) / max(len(m2), 1)
print(f"Raw MODIS:VIIRS detection ratio : {ratio_raw:.2f}:1")
print(f"Gridded MODIS:VIIRS cell-day ratio: {ratio_cell:.2f}:1")
print(f"Raw->grid reduction : {100*(1-len(fires_gridded)/len(fires)):.1f}%")
print(f"Grid->dedup reduction : {100*(1-len(fires_dedup)/len(fires_gridded)):.1f}%")


In [ ]:
fires.to_parquet('fires_unified.parquet', index=False)
provenance = {
    'source_modis': MODIS_PATH,
    'source_viirs': VIIRS_PATH,
    'layers': [0, 1, 2, 3],
    'filters': ['confidence_rank >= 2', 'type == 0'],
    'pixel_area_km2': True,
}
import json as _json
with open('fires_unified.meta.json', 'w') as fh:
    _json.dump(provenance, fh, indent=2)

print('Wrote fires_unified.parquet + fires_unified.meta.json')
